# Retaj — Sales Demand Forecasting Benchmark

Reproducible benchmark for the TechZone forecasting slice (`RET-ML`).

1. Builds the gapless feature matrix from the daily time-series store.
2. Enforces a strict chronological split (fit / calibration / test) — no shuffling, no leakage.
3. Trains LightGBM (or sklearn fallback) quantile regressors on the scale-normalised target.
4. Calibrates the 95% band with conformalised quantile regression.
5. Benchmarks the model against SARIMA and Prophet baselines.
6. Logs metrics, the benchmark table and the model artifact to MLflow.

Run with: `jupyter lab apps/ml/notebooks/forecast_benchmark.ipynb`

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pnpm-workspace.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd

from apps.ml.src.forecast import feature_builder
from apps.ml.src.forecast.mock_store import daily_series, list_products
from apps.ml.src.forecast.train_forecast import run_training

pd.set_option('display.max_columns', 40)
print('repo root:', REPO_ROOT)

## 1. Gapless feature matrix

Every SKU gets one row per calendar day (zero-filled), and the feature matrix is built from that gapless series.

In [ ]:
frames = []
for product in list_products():
    rows = daily_series(product['product_id'], days=180)
    frame = feature_builder.build_feature_frame(pd.DataFrame(rows))
    frame = feature_builder.supervised_rows(frame)
    frame = feature_builder.add_scaled_target(frame)
    frames.append(frame)
dataset = pd.concat(frames, ignore_index=True)

gaps = dataset.groupby(['tenant_id', 'product_id'])['report_date'].nunique()
print('rows:', len(dataset), '| distinct days per (tenant, product):', gaps.unique())
assert gaps.nunique() == 1, 'gapless series violated'
dataset.head()

## 2. Strict chronological split + no-leakage assertion

In [ ]:
train, test = feature_builder.time_based_split(dataset, train_ratio=0.8)
fit, calibration = feature_builder.time_based_split(train, train_ratio=0.8)
feature_builder.assert_no_leakage(fit, calibration)
feature_builder.assert_no_leakage(train, test)
print('fit       :', fit['report_date'].min(), '->', fit['report_date'].max(), len(fit), 'rows')
print('calibration:', calibration['report_date'].min(), '->', calibration['report_date'].max(), len(calibration), 'rows')
print('test      :', test['report_date'].min(), '->', test['report_date'].max(), len(test), 'rows')
assert fit['report_date'].max() < calibration['report_date'].min() < test['report_date'].min()

## 3. Training, benchmark and MLflow logging

`run_training` fits the quantile models, calibrates the band, benchmarks SARIMA/Prophet and logs the run to MLflow.

In [ ]:
metrics = run_training(quick=False, promote=True)
metrics

## 4. Benchmark table (>= 3 model families)

In [ ]:
benchmark = pd.DataFrame(metrics['benchmark'])
benchmark

## 5. Per-SKU quality and aggregate band coverage

In [ ]:
per_product = pd.DataFrame(metrics['per_product'])
per_product['abs_error'] = (per_product['actual'] - per_product['predicted']).abs()
per_product['wape_pct'] = per_product['abs_error'] / per_product['actual'] * 100
print('per-day 95% band coverage :', metrics['coverage_95'], '%')
print('total  95% band coverage :', metrics['coverage_95_total'], '%')
per_product

## 6. MLflow run

The run (metrics + benchmark table + model artifact) is stored in MLflow; set `MLFLOW_URI` to point at the compose tracking server.

In [ ]:
import mlflow

uri = os.getenv('MLFLOW_URI') or os.getenv('MLFLOW_TRACKING_URI') or f"sqlite:///{(REPO_ROOT / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(uri)
client = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name('techzone_forecast_demo')
runs = client.search_runs(experiment.experiment_id, order_by=['attributes.start_time DESC'], max_results=5) if experiment else []
print('tracking uri:', uri)
for run in runs:
    print(run.info.run_id, run.info.status, {k: round(v, 3) for k, v in run.data.metrics.items() if not k.startswith('bench_')})